In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("train.csv")

# Drop high-cardinality / unused columns
df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace=True)

X = df.drop(columns=["Survived"])
y = df["Survived"]

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=2,
    stratify=y
)

In [4]:
numerical_features = ["Age", "Fare", "Pclass", "SibSp", "Parch"]
categorical_features = ["Embarked", "Sex"]


In [5]:
# Numeric: median/mean imputation + scaling
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical: most_frequent/constant + OneHotEncoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combine using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"   # keep only selected columns
)

In [6]:
# ==========================================================
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, solver="lbfgs"))
])


In [7]:
param_grid = {
    "preprocessor__num__imputer__strategy": ["mean", "median"],
    "preprocessor__cat__imputer__strategy": ["most_frequent", "constant"],
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

# Better CV (Stratified)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=2)

grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

In [8]:
grid_search.fit(X_train, y_train)

print("\n✅ Best Params:")
print(grid_search.best_params_)

print(f"\n✅ Best CV Accuracy: {grid_search.best_score_:.3f}")


✅ Best Params:
{'classifier__C': 0.1, 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}

✅ Best CV Accuracy: 0.800


In [9]:
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test)

print(f"\n✅ Test Accuracy: {accuracy_score(y_test, y_pred_test):.3f}")
print("\n✅ Classification Report:\n")
print(classification_report(y_test, y_pred_test))



✅ Test Accuracy: 0.799

✅ Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.86      0.84       110
           1       0.76      0.70      0.73        69

    accuracy                           0.80       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.80      0.80      0.80       179



In [10]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values("mean_test_score", ascending=False)

print("\n✅ Top GridSearch Results:\n")
print(cv_results[
    [
        "param_classifier__C",
        "param_preprocessor__cat__imputer__strategy",
        "param_preprocessor__num__imputer__strategy",
        "mean_test_score",
        "mean_train_score"
    ]
].head(10))


✅ Top GridSearch Results:

    param_classifier__C param_preprocessor__cat__imputer__strategy  \
6                   0.1                                   constant   
7                   0.1                                   constant   
4                   0.1                              most_frequent   
13                 10.0                              most_frequent   
17                100.0                              most_frequent   
19                100.0                                   constant   
15                 10.0                                   constant   
5                   0.1                              most_frequent   
16                100.0                              most_frequent   
12                 10.0                              most_frequent   

   param_preprocessor__num__imputer__strategy  mean_test_score  \
6                                        mean         0.800411   
7                                      median         0.800411   
4  